# Eddy deformation, ellipse orientation and tilt

**Primary question:** does surface elongation relate to measured tilt magnitude, and does tilt have a preferred direction relative to the ellipse? Parallel alignment is a possibility, not an assumed outcome. AE and CE are analysed separately.

**Secondary question:** do ellipse geometries at 50, 100, 200 and 300 m show stronger relationships? Every depth uses the **same existing whole-column `TiltDis` and `TiltDir`**; this notebook never recomputes tilt.

Run on Katana with the modular Parquet outputs. Start in this folder, the parent analysis folder, or the repository root. No new model-data cache is needed. Results are exploratory associations, not causal tests or out-of-sample predictions.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate this workflow without a machine-specific repository path.
candidates = [Path.cwd(), Path.cwd() / 'ellipse_tilt_analysis',
              Path.cwd() / 'seacofs_eddy_tilt_analysis' / 'ellipse_tilt_analysis',
              Path.cwd() / 'UNSW-PhD' / 'seacofs_eddy_tilt_analysis' / 'ellipse_tilt_analysis']
HERE = next((p.resolve() for p in candidates if (p / 'ellipse_tilt_tools.py').exists()), None)
if HERE is None:
    raise FileNotFoundError('Launch from ellipse_tilt_analysis, its parent, or the UNSW-PhD repository root.')
sys.path.insert(0, str(HERE.parent))
sys.path.insert(0, str(HERE))
import seacofs_tilt_tools as tilt
import ellipse_tilt_tools as et

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.dpi': 300})
DEPTHS = (0, 50, 100, 200, 300)
RUN_DEPTHS = True
# Matches compute_weighted_tilt's bearing_offset_deg in config/local.yaml.
BEARING_OFFSET_DEG = 20.0
MIN_DIRECTION_AR = 1.1
MIN_DIRECTION_TILT_KM = 5.0
MAX_AR = 5.0
MAX_DEPTH_BRACKET_M = 100.0
NEAREST_TOLERANCE_M = 25.0
MIN_DAYS_PER_EDDY = 5
MIN_EDDIES = 10
N_BOOT = 1000
SEED = 42
SAVE_OUTPUTS = False
OUTPUT = HERE / 'outputs'

paths = tilt.Paths()  # Override individual paths here if your files moved.
stat_kwargs = dict(n_boot=N_BOOT, seed=SEED, min_days=MIN_DAYS_PER_EDDY, min_eddies=MIN_EDDIES)
figures = {}
def show_figure(name, fig):
    figures[name] = fig
    plt.show()

## 1. Geometry and conventions

The fitted matrix defines $\rho^2 = \mathbf{x}^T Q\mathbf{x}$. The **smallest eigenvalue** identifies the longest axis; major/minor axis ratio is $\sqrt{\lambda_{max}/\lambda_{min}}$. Invalid/non-positive-definite matrices are rejected. Exactly circular ellipses have no defined major-axis direction.

The ML helper `_major_axis_encoding` uses doubled mathematical angles. Here we instead form a bearing directly from the major eigenvector `(vx, vy)` as `atan2(vx, vy) + BEARING_OFFSET_DEG`, matching `core/tilt.py`'s tilt bearing. Eigenvector sign is irrelevant because the axis is modulo 180°. Do not apply an additional grid rotation.

The signed tilt–axis offset lies in [-90°, 90°); its absolute value lies in [0°, 90°]. `cos(2 offset)` is +1 for parallel and −1 for perpendicular. Its mean alone can miss intermediate or multimodal preferences, so retain the full histogram and `sin(2 offset)` too. Neither statistic distinguishes the two ends of a major axis.

Thresholds are exploratory defaults: AR ≥1.1 and tilt ≥5 km for directional analyses, AR ≤5 for all analyses (matching the surface QC ceiling). Magnitude analyses retain circular shapes and zero tilt. Sensitivity tests vary the directional thresholds below.

In [ ]:
grid = tilt.load_grid(paths.grid, paths.z_r)
surface, _ = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
if surface.duplicated(['Eddy', 'Day']).any():
    raise ValueError('Duplicate surface/tilt join keys; resolve before analysis.')
# Core means match the case-study PV context; regimes are descriptive covariates.
surface = tilt.add_pv_gradient_terms(surface, grid, core_mean=True)
print(f'Surface input: {len(surface):,} eddy-days; {surface.Eddy.nunique():,} eddies')

# Independent geometric check: synthetic x, y and 45-degree major axes.
check = pd.DataFrame({'q11': [.25, 1, .625, 1], 'q12': [0, 0, -.375, 0],
                      'q22': [1, .25, .625, 1]})
check = et.ellipse_geometry(check, BEARING_OFFSET_DEG)
expected = (np.array([90, 0, 45]) + BEARING_OFFSET_DEG) % 180
assert np.allclose(et.axial_difference(check.MajorBearing.iloc[:3], expected), 0)
assert np.allclose(check.AxisRatio.iloc[:3], 2)
assert np.isnan(check.MajorBearing.iloc[3])
display(check)

## 2. Surface and depth sampling; coverage audit

Load only the necessary vertical-profile columns. Use exact levels where available; otherwise linearly interpolate **Q entries**, not axis angles or axis ratios. Adjacent bracketing matrices must both be positive definite and no more than 100 m apart. No extrapolation, bridging invalid levels or silent averaging of duplicate profile rows is allowed. Depth is positive down.

Interpolating Q can reduce apparent elongation when axes rotate between levels; nearest-level sensitivity is included later. Surface Q is processed/smoothed by the upstream pipeline, whereas subsurface fits may have different smoothing and uncertainty. Any apparent depth optimum must be interpreted with that difference in mind.

In [ ]:
profile_columns = ['Eddy', 'Day', 'Depth', 'q11', 'q12', 'q22']
if RUN_DEPTHS:
    if Path(paths.vert).suffix.lower() == '.parquet':
        profiles = pd.read_parquet(paths.vert, columns=profile_columns)
    else:
        profiles = tilt.load_vert(paths, dic_form=False)
        if not isinstance(profiles, pd.DataFrame):
            raise TypeError('Use the modular long-form profiles.parquet, not a legacy dictionary.')
        profiles = profiles[profile_columns]
    profiles = profiles.merge(surface[['Eddy', 'Day']], on=['Eddy', 'Day'],
                              how='inner', validate='many_to_one')
    sampled = et.sample_depth_geometry(profiles, DEPTHS[1:], max_gap_m=MAX_DEPTH_BRACKET_M)
else:
    sampled = pd.DataFrame(columns=['Eddy', 'Day', 'ShapeDepth', 'q11', 'q12', 'q22',
                                    'DepthLower', 'DepthUpper', 'DepthMethod'])
all_shapes = et.build_analysis_table(surface, sampled, BEARING_OFFSET_DEG)
magnitude = et.select_rows(all_shapes, max_ar=MAX_AR)
direction = et.select_rows(all_shapes, directional=True, min_ar=MIN_DIRECTION_AR,
                           min_tilt=MIN_DIRECTION_TILT_KM, max_ar=MAX_AR)
coverage = []
for label, frame in [('Available geometry', all_shapes), ('Magnitude QC', magnitude),
                     ('Directional QC', direction)]:
    counts = frame.groupby(['ShapeDepth', 'Cyc']).agg(observations=('Day', 'size'), eddies=('Eddy', 'nunique'))
    coverage.append(counts.assign(sample=label).reset_index())
coverage = pd.concat(coverage, ignore_index=True)
display(coverage)
display(pd.crosstab(all_shapes.ShapeDepth, all_shapes.DepthMethod))
if 'AR' in surface:
    audit = all_shapes.loc[all_shapes.ShapeDepth.eq(0), ['AR', 'AxisRatio']].dropna()
    print('Max absolute difference: stored AR versus Q-derived AR:',
          (audit.AR-audit.AxisRatio).abs().max())
print('Invalid Q matrices:', int((~all_shapes.Q_valid).sum()))

## 3. Surface elongation versus tilt distance

Each scatter point is one eddy's median AR and median tilt over the same available days. Binned lines and interquartile shading are descriptive. Formal summaries require at least five valid days per eddy and ten eddies per group.

Between-eddy Spearman correlation asks whether eddies with larger typical AR also have larger typical tilt. The within-eddy slope de-means AR and tilt within each eddy and gives each eddy equal total weight; units are km of tilt per unit AR. Its bootstrap resamples whole eddies, retaining serial dependence. These are different estimands and need not agree.

In [ ]:
show_figure('01_surface_magnitude', et.plot_magnitude(magnitude))
mag_stats = et.magnitude_summary(magnitude, **stat_kwargs)
display(mag_stats.loc[mag_stats.ShapeDepth.eq(0)])

## 4. Surface ellipse–tilt directional alignment

Every eddy has equal total weight in each histogram. The dashed line is the uniform-angle reference (5.56% per 5° bin), **not an independence test**: common geographic orientations can produce apparent alignment without direct coupling.

Confidence intervals below use one mean per eddy. Positive mean cosine favours parallel over perpendicular alignment; negative favours perpendicular. Inspect the distribution for oblique peaks or mixtures, even if mean cosine and sine are near zero. Mean absolute angle of 45° alone does not establish uniformity.

In [ ]:
show_figure('02_surface_alignment', et.plot_alignment(direction))
dir_stats = et.alignment_summary(direction, **stat_kwargs)
display(dir_stats.loc[dir_stats.ShapeDepth.eq(0)])
show_figure('03_surface_alignment_by_elongation', et.plot_interaction(direction))

## 5. Regional and PV-regime context

Upstream includes S1/U1/U2; downstream includes S2/D1/D2. PV categories use the natural-log topographic/planetary ratio: planetary <−log(2), topographic >log(2), otherwise mixed. These categories describe sampled conditions, not proven dynamical control.

Repeat both relationships within each group. An eddy can contribute to multiple sectors/regimes on different days, so these rows are not independent comparisons. Group-specific intervals are pointwise, with no multiple-testing correction. Sparse groups return missing estimates rather than unstable intervals.

In [ ]:
context_tables = {}
for context in ['Sector', 'PVRegime']:
    groups = ('ShapeDepth', 'Cyc', context)
    m = et.magnitude_summary(magnitude.loc[magnitude.ShapeDepth.eq(0)], groups=groups, **stat_kwargs)
    d = et.alignment_summary(direction.loc[direction.ShapeDepth.eq(0)], groups=groups, **stat_kwargs)
    context_tables[context + '_magnitude'] = m
    context_tables[context + '_direction'] = d
    print(context)
    display(m)
    display(d.loc[d.metric.eq('AlignmentCos2')])

## 6. Within-eddy rotation and deformation changes

The earlier within-eddy magnitude slope tests whether departures from an eddy's typical shape accompany departures from its typical tilt. Here use consecutive qualified observations at most one day apart; excluded days are not bridged.

Tilt turns are treated axially for comparison with ellipse rotation. This intentionally ignores 180° reversals of the tilt arrow. Positive `TurnAgreement = cos[2(tilt turn − axis turn)]` means similar turns, but two nearly stationary directions also agree. Therefore summarise only pairs with at least 5° of **ellipse-axis rotation**; vary that threshold if this becomes a main result. Fast rotations ≥90° between observations are aliased and cannot be resolved.

In [ ]:
pairs = et.rotation_pairs(direction.loc[direction.ShapeDepth.eq(0)], max_gap_days=1)
rotation_results = []
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (cyc, colour) in zip(axes, et.COLOURS.items()):
    g = pairs.loc[pairs.Cyc.eq(cyc) & pairs.AxisTurn.abs().ge(5)].copy()
    per_eddy = g.groupby('Eddy').agg(agreement=('TurnAgreement', 'mean'), n=('Day', 'size'))
    per_eddy = per_eddy.loc[per_eddy.n >= MIN_DAYS_PER_EDDY]
    estimate, low, high = et.bootstrap_stat(per_eddy[['agreement']].to_numpy(), np.mean,
                                           N_BOOT, SEED, MIN_EDDIES)
    rotation_results.append(dict(Cyc=cyc, eddies=len(per_eddy), estimate=estimate, low=low, high=high))
    if len(g):
        weights = et.equal_eddy_weights(g)
        hb = ax.hexbin(g.AxisTurn, g.TiltTurn, C=100 * weights / weights.sum(),
                       reduce_C_function=np.sum, extent=(-90, 90, -90, 90),
                       gridsize=30, mincnt=1, cmap='Greys', vmin=0)
        fig.colorbar(hb, ax=ax, label='Eddy-equal mass per hexagon (%)', shrink=.8)
    ax.plot([-90, 90], [-90, 90], '--', color=colour)
    ax.set(title=f'{cyc}: rotating-axis pairs', xlabel='Signed ellipse-axis turn (°)',
           xlim=(-90,90), ylim=(-90,90))
axes[0].set_ylabel('Signed axial tilt turn (°)')
fig.tight_layout()
show_figure('04_surface_rotation', fig)
rotation_results = pd.DataFrame(rotation_results)
display(rotation_results)

## 7. Depth comparison: available and matched eddy-days

First show all available observations at each depth. Then require the **same eddy-days at all five depths**, separately for magnitude QC and directional QC. This distinguishes depth dependence from selection of deeper-reaching eddies. The matched sample is selective and is not representative of all eddies; report its coverage.

Pointwise CI overlap/non-overlap is not a paired test of differences between depths. Do not select a “best depth” on these exploratory estimates alone.

In [ ]:
active_depths = DEPTHS if RUN_DEPTHS else (0,)
matched_magnitude = et.matched_depth_sample(magnitude, active_depths)
matched_direction = et.matched_depth_sample(direction, active_depths)
matched_mag_stats = et.magnitude_summary(matched_magnitude, **stat_kwargs)
matched_dir_stats = et.alignment_summary(matched_direction, **stat_kwargs)
print('Matched magnitude eddy-days:', len(matched_magnitude) // len(active_depths))
print('Matched direction eddy-days:', len(matched_direction) // len(active_depths))
for name, table in [('Available', mag_stats), ('Matched', matched_mag_stats)]:
    print(name)
    display(table)
    show_figure(name.lower() + '_depth_magnitude', et.plot_depth_summary(table, 'Between-eddy Spearman'))
for name, table in [('Available', dir_stats), ('Matched', matched_dir_stats)]:
    show_figure(name.lower() + '_depth_alignment', et.plot_depth_summary(table, 'AlignmentCos2'))
    display(table)
if RUN_DEPTHS:
    for depth in DEPTHS[1:]:
        show_figure(f'alignment_{depth}m', et.plot_alignment(direction, depth))

## 8. Sensitivity: near-circular eddies, small tilts and depth interpolation

Repeat surface alignment for AR thresholds 1.05/1.1/1.2/1.5 and tilt thresholds 0/5/10 km. Even the zero-threshold test excludes exactly zero tilt, whose direction is undefined. Record sample size alongside each result.

Nearest-depth sensitivity uses the same vertical range and a 25 m tolerance, never extrapolates, and compares **identical eddy-day-depth keys** with interpolated sampling. Inspect `DepthLower/DepthUpper` to see actual selected levels. If no common qualified observations exist, empty tables are expected.

In [ ]:
sensitivity = []
for ar in (1.05, 1.1, 1.2, 1.5):
    for minimum_tilt in (0, 5, 10):
        g = et.select_rows(all_shapes.loc[all_shapes.ShapeDepth.eq(0)], directional=True,
                           min_ar=ar, min_tilt=minimum_tilt, max_ar=MAX_AR)
        g = g.loc[g.TiltDis.gt(0)]
        result = et.alignment_summary(g, **stat_kwargs)
        sensitivity.append(result.assign(min_ar=ar, min_tilt_km=minimum_tilt))
sensitivity = pd.concat(sensitivity, ignore_index=True)
display(sensitivity.loc[sensitivity.metric.eq('AlignmentCos2')])

nearest_comparison = {}
if RUN_DEPTHS:
    nearest_q = et.sample_depth_geometry(profiles, DEPTHS[1:], method='nearest',
                                         nearest_tolerance_m=NEAREST_TOLERANCE_M)
    nearest_all = et.build_analysis_table(surface, nearest_q, BEARING_OFFSET_DEG)
    for outcome in ('magnitude', 'direction'):
        directional = outcome == 'direction'
        a = et.select_rows(all_shapes, directional=directional, min_ar=MIN_DIRECTION_AR,
                           min_tilt=MIN_DIRECTION_TILT_KM, max_ar=MAX_AR)
        b = et.select_rows(nearest_all, directional=directional, min_ar=MIN_DIRECTION_AR,
                           min_tilt=MIN_DIRECTION_TILT_KM, max_ar=MAX_AR)
        keys = ['Eddy', 'Day', 'ShapeDepth']
        common = a[keys].merge(b[keys], on=keys, validate='one_to_one')
        summarise = et.alignment_summary if directional else et.magnitude_summary
        for label, frame in [('interpolated', a), ('nearest', b)]:
            use = frame.merge(common, on=keys, validate='one_to_one')
            result = summarise(use, **stat_kwargs)
            nearest_comparison[outcome + '_' + label] = result
            print(outcome, label)
            display(result)

## 9. Interpretation and export

- Report AE and CE separately and distinguish the surface hypothesis from exploratory depth comparisons.
- Stronger typical elongation and positive within-eddy slopes together are more informative than a pooled eddy-day correlation alone. They still do not establish causation.
- Inspect full alignment distributions. Opposing or intermediate modes can cancel in mean trigonometric summaries.
- Context, threshold and sampling stability matter. Alignment with both topography and the ellipse may reflect shared environmental orientation.
- Shape and tilt come from related velocity fits. Common fitting errors, surface-Q smoothing and the tilt pipeline's temporal smoothing can induce associations or affect daily-rotation results.
- Tilt distance depends on the measured vertical span. A later adjusted analysis should account for fitted depth range, radius, age, latitude and environment before interpreting elongation as an independent driver.
- Bootstrap intervals assume independence between eddies; they do not account for correlations among neighbouring eddies or multiple exploratory comparisons.
- Compare with the earlier ML ellipse-feature ablation only as complementary evidence: predictive feature importance and these descriptive associations answer different questions.

No output files are written unless `SAVE_OUTPUTS=True`. The saved settings and coverage tables accompany the figures; the notebook is distributed with outputs cleared because the production data live on Katana.

In [ ]:
if SAVE_OUTPUTS:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    tables = dict(coverage=coverage, magnitude=mag_stats, direction=dir_stats,
                  matched_magnitude=matched_mag_stats, matched_direction=matched_dir_stats,
                  rotation=rotation_results, threshold_sensitivity=sensitivity,
                  **context_tables, **nearest_comparison)
    for name, table in tables.items():
        table.to_csv(OUTPUT / f'{name}.csv', index=False)
    for name, fig in figures.items():
        fig.savefig(OUTPUT / f'{name}.pdf', bbox_inches='tight')
        fig.savefig(OUTPUT / f'{name}.png', dpi=600, bbox_inches='tight')
    settings = dict(depths=active_depths, bearing_offset_deg=BEARING_OFFSET_DEG,
                    min_direction_ar=MIN_DIRECTION_AR, min_direction_tilt_km=MIN_DIRECTION_TILT_KM,
                    max_ar=MAX_AR, max_depth_bracket_m=MAX_DEPTH_BRACKET_M,
                    nearest_tolerance_m=NEAREST_TOLERANCE_M, n_boot=N_BOOT, seed=SEED,
                    min_days_per_eddy=MIN_DAYS_PER_EDDY, min_eddies=MIN_EDDIES,
                    paths={k: str(v) for k, v in vars(paths).items()})
    (OUTPUT / 'settings.json').write_text(json.dumps(settings, indent=2))
    print('Saved to', OUTPUT)
else:
    print('Review the figures and sensitivity tables before choosing final publication panels.')